# TÁCH TỪ TIẾNG VIỆT BẰNG FMM, BMM VÀ HMM–VITERBI
## Notebook tuần 4

Notebook ghi lại quá trình hoàn thiện dự án từ kết quả tuần 3 và các nhiệm vụ tuần 4: tách mã nguồn thành module, kiểm thử Unit Test, tối ưu giới hạn độ dài từ `L`, đo thời gian và thực nghiệm với 3 Seed.

**Cấu trúc mã nguồn trong môi trường Google Colab:**
- `dataset.csv`: dữ liệu gồm 120 câu đã gán nhãn.
- `fmm.py`, `bmm.py`, `hmm.py`: các module thuật toán.
- `test_fmm_bmm.py`, `test_viterbi.py`: các file kiểm thử.
- `run_experiments.py`: chương trình thực nghiệm 3 Seed.

> Trước khi chạy notebook, hãy tải `dataset.csv` lên thư mục làm việc của Google Colab. Chạy các ô theo thứ tự từ trên xuống dưới.

## 1. Chuẩn bị thư viện và dữ liệu

In [1]:
import math
import time
import importlib
import pandas as pd
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

print("Đã nạp các thư viện cần thiết.")

Đã nạp các thư viện cần thiết.


In [3]:
# Đọc dữ liệu
data = pd.read_csv("dataset.csv", encoding="utf-8-sig")

required_columns = {"sentence", "segmented"}
if not required_columns.issubset(data.columns):
    raise ValueError("Dataset phải có hai cột: sentence và segmented")

print("Số câu:", len(data))
display(data.head())

Số câu: 120


,sentence,segmented
0,Tôi đang học bài,Tôi đang học bài
1,Hôm nay trời rất đẹp,Hôm_nay trời rất đẹp
2,Tôi thích nghe nhạc,Tôi thích nghe nhạc
3,Mẹ đang nấu cơm,Mẹ đang nấu cơm
4,Bố đi làm từ sáng,Bố đi làm từ sáng


In [4]:
# Kiểm tra số tiếng trong câu gốc có khớp với số tiếng sau khi tách nhãn
alignment_errors = []

for idx, row in data.iterrows():
    original = row["sentence"].split()
    reconstructed = []
    for word in str(row["segmented"]).split():
        reconstructed.extend(word.split("_"))

    if original != reconstructed:
        alignment_errors.append(idx)

print("Số dòng không khớp:", len(alignment_errors))
if alignment_errors:
    print("Các dòng lỗi:", alignment_errors)
else:
    print("Dữ liệu khớp về số tiếng và thứ tự tiếng.")

Số dòng không khớp: 0
Dữ liệu khớp về số tiếng và thứ tự tiếng.


## 2. Chia Train/Test và xây dựng từ điển

Sử dụng cách chia 80% Train và 20% Test với `random_state=42` để tái lập kết quả cơ sở. Từ điển FMM/BMM chỉ được xây dựng từ tập Train để tránh sử dụng thông tin của tập Test.

In [5]:
train, test = train_test_split(data, test_size=0.2, random_state=42)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

dictionary = set()
for segmented in train["segmented"]:
    dictionary.update(str(segmented).split())

L = max((len(word.split("_")) for word in dictionary), default=1)

print("Số câu Train:", len(train))
print("Số câu Test:", len(test))
print("Số từ trong từ điển:", len(dictionary))
print("Độ dài từ lớn nhất L:", L)

Số câu Train: 96
Số câu Test: 24
Số từ trong từ điển: 302
Độ dài từ lớn nhất L: 3


## 3. Tách mã nguồn thành các module

Các hàm được lưu riêng trong ba file Python. Ở các ô dưới đây, `%%writefile` tạo hoặc ghi nội dung module trong thư mục làm việc của Colab.

In [6]:
%%writefile fmm.py

def fmm(sentence, dictionary, L=None):
    syllables = sentence.split()
    result = []
    i = 0

    if L is None:
        L = max((len(word.split("_")) for word in dictionary), default=1)

    while i < len(syllables):
        found = False

        for j in range(min(len(syllables), i + L), i, -1):
            candidate = "_".join(syllables[i:j])

            if candidate in dictionary:
                result.append(candidate)
                i = j
                found = True
                break

        if not found:
            result.append(syllables[i])
            i += 1

    return " ".join(result)

Writing fmm.py


In [7]:
%%writefile bmm.py

def bmm(sentence, dictionary, L=None):
    syllables = sentence.split()
    result = []
    i = len(syllables) - 1

    if L is None:
        L = max((len(word.split("_")) for word in dictionary), default=1)

    while i >= 0:
        found = False

        for j in range(max(0, i - L + 1), i + 1):
            candidate = "_".join(syllables[j:i + 1])

            if candidate in dictionary:
                result.insert(0, candidate)
                i = j - 1
                found = True
                break

        if not found:
            result.insert(0, syllables[i])
            i -= 1

    return " ".join(result)

Writing bmm.py


In [8]:
%%writefile hmm.py

import math
from collections import Counter, defaultdict


def train_hmm(train, alpha=1):
    states = ["B", "I"]

    # Đếm trạng thái bắt đầu, chuyển trạng thái và phát xạ
    start_counts = Counter()
    transition_counts = defaultdict(Counter)
    emission_counts = defaultdict(Counter)
    state_counts = Counter()
    vocabulary = set()

    for sentence, segmented in zip(train["sentence"], train["segmented"]):
        syllables = sentence.split()
        labels = []

        for word in segmented.split():
            parts = word.split("_")
            labels.extend(["B"] + ["I"] * (len(parts) - 1))

        if len(syllables) != len(labels):
            raise ValueError("Số tiếng và số nhãn không khớp!")

        if not labels:
            continue

        start_counts[labels[0]] += 1

        for i, (syllable, state) in enumerate(zip(syllables, labels)):
            vocabulary.add(syllable)
            emission_counts[state][syllable] += 1
            state_counts[state] += 1

            if i > 0:
                transition_counts[labels[i - 1]][state] += 1

    vocabulary.add("<UNK>")

    start_prob = {}
    transition_prob = {}
    emission_prob = {}

    # Xác suất bắt đầu
    for state in states:
        start_prob[state] = (start_counts[state] + alpha) / (
            sum(start_counts.values()) + alpha * len(states)
        )

    # Xác suất chuyển trạng thái
    for previous_state in states:
        transition_prob[previous_state] = {}
        outgoing_total = sum(transition_counts[previous_state].values())

        for state in states:
            transition_prob[previous_state][state] = (
                transition_counts[previous_state][state] + alpha
            ) / (outgoing_total + alpha * len(states))

    # Xác suất phát xạ
    for state in states:
        emission_prob[state] = {}

        for syllable in vocabulary:
            emission_prob[state][syllable] = (
                emission_counts[state][syllable] + alpha
            ) / (state_counts[state] + alpha * len(vocabulary))

    model = {
        "states": states,
        "vocabulary": vocabulary,
        "start_prob": start_prob,
        "transition_prob": transition_prob,
        "emission_prob": emission_prob
    }

    return model


def viterbi(sentence, model):
    syllables = sentence.split()
    n = len(syllables)

    if n == 0:
        return ""

    states = model["states"]
    vocabulary = model["vocabulary"]
    start_prob = model["start_prob"]
    transition_prob = model["transition_prob"]
    emission_prob = model["emission_prob"]

    dp = [{} for _ in range(n)]
    backpointer = [{} for _ in range(n)]

    # Khởi tạo
    first = syllables[0] if syllables[0] in vocabulary else "<UNK>"

    for state in states:
        dp[0][state] = (
            math.log(start_prob[state])
            + math.log(emission_prob[state][first])
        )

    # Quy hoạch động
    for i in range(1, n):
        word = syllables[i] if syllables[i] in vocabulary else "<UNK>"

        for state in states:
            best_score = float("-inf")
            best_previous = None

            for previous_state in states:
                score = (
                    dp[i - 1][previous_state]
                    + math.log(transition_prob[previous_state][state])
                    + math.log(emission_prob[state][word])
                )

                if score > best_score:
                    best_score = score
                    best_previous = previous_state

            dp[i][state] = best_score
            backpointer[i][state] = best_previous

    # Chọn trạng thái cuối và truy vết
    last_state = max(dp[-1], key=dp[-1].get)
    best_path = [last_state]

    for i in range(n - 1, 0, -1):
        last_state = backpointer[i][last_state]
        best_path.append(last_state)

    best_path.reverse()

    # B ghép bằng dấu cách, I ghép bằng dấu gạch dưới
    result = []

    for i, syllable in enumerate(syllables):
        if i > 0:
            result.append(" " if best_path[i] == "B" else "_")

        result.append(syllable)

    return "".join(result)

Writing hmm.py


In [9]:
from fmm import fmm
from bmm import bmm
from hmm import train_hmm, viterbi

print("Đã tạo và import thành công các module FMM, BMM và HMM.")

Đã tạo và import thành công các module FMM, BMM và HMM.


### 3.1. Kiểm tra nhanh FMM và BMM

Dùng một ví dụ đơn giản để kiểm tra hai hàm sau khi tách thành module.

In [10]:
toy_dictionary = {"học", "sinh", "học_sinh", "sinh_học"}
sentence_test = "học sinh học sinh học"

print("FMM:", fmm(sentence_test, toy_dictionary))
print("BMM:", bmm(sentence_test, toy_dictionary))

FMM: học_sinh học_sinh học
BMM: học sinh_học sinh_học


## 4. Kiểm thử Unit Test

Xây dựng 3 kiểm thử:
1. FMM với câu `học sinh học sinh học`.
2. BMM với cùng câu trên.
3. Viterbi với ví dụ gồm 3 tiếng `x y z`, đường đi tối ưu mong đợi là `B → I → B`, tương ứng kết quả `x_y z`.

In [11]:
%%writefile test_fmm_bmm.py

import unittest
from fmm import fmm
from bmm import bmm


class TestFMMBMM(unittest.TestCase):

    def setUp(self):
        self.dictionary = {
            "học",
            "sinh",
            "học_sinh",
            "sinh_học"
        }

        self.sentence = "học sinh học sinh học"

    def test_fmm(self):
        result = fmm(self.sentence, self.dictionary)

        expected = "học_sinh học_sinh học"

        self.assertEqual(result, expected)

    def test_bmm(self):
        result = bmm(self.sentence, self.dictionary)

        expected = "học sinh_học sinh_học"

        self.assertEqual(result, expected)


if __name__ == "__main__":
    unittest.main(argv=["first-arg-is-ignored"], exit=False)

Writing test_fmm_bmm.py


In [12]:
%%writefile test_viterbi.py

import unittest
from hmm import viterbi


class TestViterbi(unittest.TestCase):

    def test_viterbi_three_syllables(self):
        model = {
            "states": ["B", "I"],
            "vocabulary": {"x", "y", "z"},
            "start_prob": {
                "B": 0.8,
                "I": 0.2
            },
            "transition_prob": {
                "B": {"B": 0.4, "I": 0.6},
                "I": {"B": 0.7, "I": 0.3}
            },
            "emission_prob": {
                "B": {"x": 0.5, "y": 0.2, "z": 0.3},
                "I": {"x": 0.1, "y": 0.6, "z": 0.3}
            }
        }

        result = viterbi("x y z", model)

        expected = "x_y z"

        self.assertEqual(result, expected)


if __name__ == "__main__":
    unittest.main(argv=["first-arg-is-ignored"], exit=False)

Writing test_viterbi.py


In [13]:
!python -m unittest test_fmm_bmm test_viterbi

...
----------------------------------------------------------------------
Ran 3 tests in 0.001s

OK


Kết quả mong đợi là `Ran 3 tests` và `OK`. Điều này cho biết cả ba trường hợp kiểm thử đều vượt qua.

## 5. Tối ưu FMM và BMM bằng giới hạn độ dài từ L

Trước tối ưu, thuật toán thử ghép các cụm tiếng với độ dài có thể lên đến toàn bộ phần câu còn lại. Sau tối ưu, chỉ thử tối đa `L` tiếng, trong đó `L` là số tiếng của từ dài nhất trong từ điển.

Để tránh tính lại `L` cho từng câu, chương trình tính `L` một lần rồi truyền vào hàm FMM/BMM.

In [14]:
# Nạp lại module sau khi ghi file
import fmm as fmm_module
import bmm as bmm_module
importlib.reload(fmm_module)
importlib.reload(bmm_module)

# Hàm FMM/BMM phiên bản cũ để so sánh
def fmm_old(sentence, dictionary):
    syllables = sentence.split()
    result = []
    i = 0
    while i < len(syllables):
        found = False
        for j in range(len(syllables), i, -1):
            candidate = "_".join(syllables[i:j])
            if candidate in dictionary:
                result.append(candidate)
                i = j
                found = True
                break
        if not found:
            result.append(syllables[i])
            i += 1
    return " ".join(result)

def bmm_old(sentence, dictionary):
    syllables = sentence.split()
    result = []
    i = len(syllables) - 1
    while i >= 0:
        found = False
        for j in range(0, i + 1):
            candidate = "_".join(syllables[j:i + 1])
            if candidate in dictionary:
                result.insert(0, candidate)
                i = j - 1
                found = True
                break
        if not found:
            result.insert(0, syllables[i])
            i -= 1
    return " ".join(result)

# Kiểm tra kết quả trước và sau tối ưu có giống nhau
sentences = test["sentence"].tolist()

for sentence in sentences:
    assert fmm_old(sentence, dictionary) == fmm_module.fmm(sentence, dictionary, L)
    assert bmm_old(sentence, dictionary) == bmm_module.bmm(sentence, dictionary, L)

print("Kết quả trước và sau tối ưu giống nhau trên tập Test.")

Kết quả trước và sau tối ưu giống nhau trên tập Test.


In [18]:
# Đo thời gian thực thi: lặp 1000 lần trên tập Test
repeat = 1000

def measure_time_old(func):
    start = time.perf_counter()
    for _ in range(repeat):
        for sentence in sentences:
            func(sentence, dictionary)
    return time.perf_counter() - start

def measure_time_optimized(func):
    start = time.perf_counter()
    for _ in range(repeat):
        for sentence in sentences:
            func(sentence, dictionary, L)
    return time.perf_counter() - start

time_fmm_old = measure_time_old(fmm_old)
time_fmm_new = measure_time_optimized(fmm_module.fmm)
time_bmm_old = measure_time_old(bmm_old)
time_bmm_new = measure_time_optimized(bmm_module.bmm)

timing_results = pd.DataFrame({
    "Thuật toán": ["FMM", "BMM"],
    "Trước tối ưu (giây)": [time_fmm_old, time_bmm_old],
    "Sau tối ưu (giây)": [time_fmm_new, time_bmm_new]
})
display(timing_results.round(6))

,Thuật toán,Trước tối ưu (giây),Sau tối ưu (giây)
0,FMM,0.454608,0.206936
1,BMM,0.417312,0.220810


### 5.1. Kết quả đo đã ghi nhận trong lần chạy trước

| Thuật toán | Trước tối ưu (giây) | Sau tối ưu (giây) |
|---|---:|---:|
| FMM | 0.956179 | 0.212317 |
| BMM | 0.422947 | 0.219032 |

FMM nhanh hơn khoảng 4,50 lần; BMM nhanh hơn khoảng 1,93 lần trong lần đo này. Thời gian có thể thay đổi tùy phiên chạy và tài nguyên máy. Khi chạy lại notebook, ưu tiên sử dụng kết quả mới hiển thị ở bảng đo phía trên.

## 6. Thực nghiệm với 3 Seed và tính F1 trung bình

Chương trình dưới đây tự chia dữ liệu thành Train/Test với ba giá trị Seed: `42`, `123`, `2026`. Ở mỗi lần chia, từ điển được tạo từ Train, HMM được huấn luyện trên Train, sau đó FMM/BMM/HMM được đánh giá trên Test.

F1 được tính theo ranh giới từ (boundary-based micro F1).

In [19]:
%%writefile run_experiments.py

import pandas as pd
from sklearn.model_selection import train_test_split

from fmm import fmm
from bmm import bmm
from hmm import train_hmm, viterbi


# Tính các vị trí ranh giới từ
def get_boundaries(segmented):
    words = segmented.split()
    boundaries = set()
    position = 0

    for word in words:
        position += len(word.split("_"))
        boundaries.add(position)

    boundaries.discard(position)
    return boundaries


# Tính Precision, Recall và F1
def evaluate(gold, predicted):
    tp = 0
    fp = 0
    fn = 0

    for true_sentence, pred_sentence in zip(gold, predicted):
        true_boundaries = get_boundaries(true_sentence)
        pred_boundaries = get_boundaries(pred_sentence)

        tp += len(true_boundaries & pred_boundaries)
        fp += len(pred_boundaries - true_boundaries)
        fn += len(true_boundaries - pred_boundaries)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return f1


# Đọc dữ liệu
df = pd.read_csv("dataset.csv")

seeds = [42, 123, 2026]

results = []

for seed in seeds:
    print("\n========== SEED", seed, "==========")

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=seed
    )

    # Tạo từ điển từ tập Train
    dictionary = set()

    for sentence in train_df["segmented"]:
        for word in sentence.split():
            dictionary.add(word)

    # Tính L
    L = max(
        (len(word.split("_")) for word in dictionary),
        default=1
    )

    # Huấn luyện HMM
    model = train_hmm(train_df)

    # Dự đoán
    gold = test_df["segmented"].tolist()

    pred_fmm = [
        fmm(sentence, dictionary, L)
        for sentence in test_df["sentence"]
    ]

    pred_bmm = [
        bmm(sentence, dictionary, L)
        for sentence in test_df["sentence"]
    ]

    pred_hmm = [
        viterbi(sentence, model)
        for sentence in test_df["sentence"]
    ]

    # Tính F1
    f1_fmm = evaluate(gold, pred_fmm)
    f1_bmm = evaluate(gold, pred_bmm)
    f1_hmm = evaluate(gold, pred_hmm)

    results.append({
        "Seed": seed,
        "FMM": f1_fmm,
        "BMM": f1_bmm,
        "HMM": f1_hmm
    })

    print("F1 FMM:", round(f1_fmm, 4))
    print("F1 BMM:", round(f1_bmm, 4))
    print("F1 HMM:", round(f1_hmm, 4))


# Tổng hợp kết quả
result_df = pd.DataFrame(results)

print("\n========== BẢNG KẾT QUẢ ==========")
print(result_df.to_string(index=False))

print("\n========== F1 TRUNG BÌNH ==========")
print("FMM:", round(result_df["FMM"].mean(), 4))
print("BMM:", round(result_df["BMM"].mean(), 4))
print("HMM:", round(result_df["HMM"].mean(), 4))

Writing run_experiments.py


In [20]:
!python run_experiments.py


========== SEED 42 ==========
F1 FMM: 0.8571
F1 BMM: 0.8571
F1 HMM: 0.8252

========== SEED 123 ==========
F1 FMM: 0.8548
F1 BMM: 0.8548
F1 HMM: 0.8325

========== SEED 2026 ==========
F1 FMM: 0.855
F1 BMM: 0.855
F1 HMM: 0.8378

========== BẢNG KẾT QUẢ ==========
 Seed      FMM      BMM      HMM
   42 0.857143 0.857143 0.825243
  123 0.854772 0.854772 0.832487
 2026 0.855019 0.855019 0.837838

========== F1 TRUNG BÌNH ==========
FMM: 0.8556
BMM: 0.8556
HMM: 0.8319


### 6.1. Kết quả thực nghiệm đã ghi nhận

| Seed | F1 FMM | F1 BMM | F1 HMM |
|---|---:|---:|---:|
| 42 | 0.8571 | 0.8571 | 0.8252 |
| 123 | 0.8548 | 0.8548 | 0.8325 |
| 2026 | 0.8550 | 0.8550 | 0.8378 |
| **F1 trung bình** | **0.8556** | **0.8556** | **0.8319** |

Trong bộ dữ liệu và cách đánh giá hiện tại, FMM và BMM có F1 trung bình bằng nhau và cao hơn HMM. Kết quả chỉ phản ánh bộ dữ liệu 120 câu đang sử dụng, không khẳng định thứ hạng này đúng với mọi dữ liệu tiếng Việt.

## 7. Tổng kết tuần 4

Các công việc đã thực hiện:
- Tách FMM, BMM và HMM thành các module Python riêng.
- Xây dựng 3 Unit Test và chạy thành công.
- Xác định `L = 3` tiếng theo từ điển thu được từ tập Train.
- Giới hạn phạm vi tìm kiếm của FMM/BMM và kiểm tra kết quả trước/sau tối ưu không thay đổi trên tập Test.
- Đo thời gian thực thi trước và sau tối ưu.
- Chạy thực nghiệm với 3 Seed và tính F1 trung bình.

**Lưu ý về cách đánh giá:** Kết quả F1 trong phần thực nghiệm 3 Seed sử dụng cách tính ranh giới từ trong `run_experiments.py`. Không so sánh trực tiếp với các số F1 tuần 3 nếu tuần 3 dùng cách tính hoặc tập đánh giá khác.

## 8. Các lệnh kiểm tra cuối cùng

Có thể chạy lại hai lệnh sau trong Colab để kiểm tra Unit Test và thực nghiệm:

```bash
!python -m unittest test_fmm_bmm test_viterbi
!python run_experiments.py
```
